# Code Generator

Using Open Source models to generate high performance C++ code from Python code.

In [36]:
# Imports

import os
import io
import sys

import openai
from dotenv import load_dotenv
from openai import OpenAI
import subprocess
from IPython.display import Markdown, display
import gradio as gr


In [37]:
# Using Open Source models from OLLAMA and GROQ:
load_dotenv(override= True)
ollama_api_key = ''
groq_api_key = os.getenv('GROQ_API_KEY')

ollama_base_url = 'http://172.23.37.63:11434/v1'
groq_base_url= 'https://api.groq.com/openai/v1'

ollama = OpenAI(base_url=ollama_base_url, api_key=ollama_api_key)
groq = OpenAI(base_url=groq_base_url, api_key=groq_api_key)

In [38]:
models = ['llama3.1:8b',
          'qwen2.5-coder:7b',
          'deepseek-coder-v2:16b',
          'llama-3.3-70b-versatile',
          'openai/gpt-oss-120b',
          'openai/gpt-oss-20b']

clients = {
    'llama3.1:8b': ollama,
    'qwen2.5-coder:7b' : ollama,
    'deepseek-coder-v2:16b' : ollama,
    'llama-3.3-70b-versatile' : groq,
    'openai/gpt-oss-120b' : groq,
    'openai/gpt-oss-20b' : groq
}

In [39]:
# System Info:

from system_info import retrieve_system_info

system_info = retrieve_system_info()
system_info

{'os': {'system': 'Windows',
  'arch': 'AMD64',
  'release': '10',
  'version': '10.0.26200',
  'kernel': '10',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': ''},
 'package_managers': ['winget'],
 'cpu': {'brand': '13th Gen Intel(R) Core(TM) i9-13900HX',
  'cores_logical': 32,
  'cores_physical': 24,
  'simd': []},
 'toolchain': {'compilers': {'gcc': '', 'g++': '', 'clang': '', 'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': ''},
  'linkers': {'ld_lld': ''}}}

In [40]:
# Compile and Run Commands for C++ Code (from previous file):

compile_command = [ r"C:\msys64\ucrt64\bin\g++.exe", "-std=c++20", "-O3", "-march=native", "-DNDEBUG", "-flto", "main_os.cpp", "-o", "main_os.exe", "-static-libstdc++", "-static-libgcc", ]

run_command = ["main_os.exe"]

## Porting Code using Open Source Models:

In [65]:
# System Prompt:
system_prompt = '''
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
Do NOT output JSON. Do NOT attempt to use any tools or function calls.
The C++ response needs to produce an identical output in the fastest possible time.
'''

# Function to create user prompt with given Python code:
def user_prompt_for(python):
    return f'''
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main_os.cpp and then compiled and executed; the compilation command is:
{compile_command}
Respond only with C++ code.
Python code to port:

```python
{python}
```
'''

In [42]:
# Function to define Messages List for LLM:
def message_for(python):
    return [{'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt_for(python)}]

In [43]:
# Function to write given output C++ code from LLM to a file:
def write_output(cpp):
    with open('main_os.cpp', 'w', encoding= 'utf-8') as f:
        f.write(cpp)

In [50]:
# Function to Port Code using OpenAI API:
def port(model, python):
    """
    Calling LLM API to with given Model to generate C++ code from given Python code.
    :param model: LLM model name
    :param python: Python Code
    """
    client = clients[model]
    response = client.chat.completions.create(model= model,
                                              messages= message_for(python))
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp', '').replace('```', '')
    write_output(reply) # Writing LLMs response to file
    return reply

In [45]:
# Python code to Port to C++ (as a string):
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200000000, 4, 1) * 4
end_time = time.time()

print(f'Result: {result:.12f}')
print(f'Execution time: {(end_time - start_time):.8f} Seconds')
"""

In [46]:
# Function to Run Python Code:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [47]:
# Running Python code:
run_python(code= pi)

'Result: 3.141592656089\nExecution time: 27.56655169 Seconds\n'

In [48]:
# Running C++ Code by following Commands from GPT above:
my_env = os.environ.copy()
my_env["PATH"] = r"C:\msys64\ucrt64\bin;" + my_env["PATH"]

def compile_and_run():
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True, env=my_env)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True, env=my_env).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True, env=my_env).stdout)
        print(subprocess.run(run_command, check=True, text=True, capture_output=True, env=my_env).stdout)

    except subprocess.CalledProcessError as e:
        print(f"An error occurred:\n{e.stderr}")

## Gradio UI to Port code from Python to C++:

In [66]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label = 'Python code:', lines= 28, value= pi)
        cpp = gr.Textbox(label= 'C++ code:', lines= 28)
    with gr.Row():
        model = gr.Dropdown(models, label= 'Select Model', value= models[0])
        convert = gr.Button('Convert Code')
    convert.click(fn= port, inputs= [model, python], outputs= [cpp])

ui.launch(inbrowser= True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [52]:
# Compile and run for different models' output:

# 1. llama31.:8b - OLLAMA:
compile_and_run()

Result: 3.141592656089
Execution time: 0.30393980 Seconds

Result: 3.141592656089
Execution time: 0.30213470 Seconds

Result: 3.141592656089
Execution time: 0.30345090 Seconds



In [54]:
# Speed-Up Compared to Python:
print(f'Speed-Up for C++ Code generated by llama:3.1:8b: {(27.56655169 / 0.30213470):.3f}')

Speed-Up for C++ Code generated by llama:3.1:8b: 91.239


In [55]:
# 2. qwen2.5-coder:7b - OLLAMA:
compile_and_run()

Result: 3.141592656089
Execution time: 0.309909500000 Seconds

Result: 3.141592656089
Execution time: 0.302486200000 Seconds

Result: 3.141592656089
Execution time: 0.301948800000 Seconds



In [56]:
# Speed-Up Compared to Python:
print(f'Speed-Up for C++ Code generated by qwen2.5-coder:7b: {(27.56655169 / 0.301948800000):.3f}')

Speed-Up for C++ Code generated by qwen2.5-coder:7b: 91.295


In [57]:
# 3. deepseek-coder-v2:16b - OLLAMA:
compile_and_run()

Result: 3.14159265609
Execution time: 0.3075745 Seconds

Result: 3.14159265609
Execution time: 0.3099321 Seconds

Result: 3.14159265609
Execution time: 0.3097626 Seconds



In [58]:
print(f'Speed-Up for C++ Code generated by deepseek-coder-v2:16b: {(27.56655169 / 0.3075745):.3f}')

Speed-Up for C++ Code generated by deepseek-coder-v2:16b: 89.626


In [59]:
# 4. llama-3.3-70b-versatile - GROQ:
compile_and_run()

Result: 3.141592656089
Execution time: 0.30636270 Seconds

Result: 3.141592656089
Execution time: 0.30394000 Seconds

Result: 3.141592656089
Execution time: 0.30836030 Seconds



In [61]:
print(f'Speed-Up for C++ Code generated by llama-3.3-70b-versatile : {(27.56655169 / 0.30394000):.3f}')

Speed-Up for C++ Code generated by llama-3.3-70b-versatile : 90.697


In [63]:
# 5. openai/gpt-oss-120b - GROQ:
compile_and_run()

Result: 3.141592656089
Execution time: 0.30190980 Seconds

Result: 3.141592656089
Execution time: 0.30314050 Seconds

Result: 3.141592656089
Execution time: 0.30367980 Seconds



In [67]:
print(f'Speed-Up for C++ Code generated by openai/gpt-oss-120b : {(27.56655169 / 0.30190980):.3f}')

Speed-Up for C++ Code generated by openai/gpt-oss-120b : 91.307


In [68]:
# 6. openai/gpt-oss-20b - GROQ:
compile_and_run()

Result: 3.141592656089
Execution time: 0.29992970 Seconds

Result: 3.141592656089
Execution time: 0.30502050 Seconds

Result: 3.141592656089
Execution time: 0.30205140 Seconds



In [69]:
print(f'Speed-Up for C++ Code generated by openai/gpt-oss-20b : {(27.56655169 / 0.29992970):.3f}')

Speed-Up for C++ Code generated by openai/gpt-oss-20b : 91.910
